# 01 · 중진공 자산거래중개장터 매물정보

| | |
|---|---|
| **목적** | 중고 설비 매물 실데이터의 EDA + 전처리. `assets.category` 매핑과 잔가율 검토의 근거 |
| **input** | `data/raw/external/중소벤처기업진흥공단_자산거래중개장터 매물정보_20251231.csv` (읽기 전용 · cp949) |
| **output** | `data-analysis/data/processed/joonggomall_assets.parquet`<br>`data-analysis/reports/joonggomall_assets_decisions.json` |

> ⛔ 원본을 복사하지 않는다. `common.py` 의 `EXTERNAL` 상수로 제자리에서 읽는다
> (CLAUDE.md 절대규칙 5 · D60).

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
from common import EXTERNAL, profile, save_decisions, save_processed, top_values

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


## 1. 개요

| 항목 | 값 |
|---|---|
| 출처 | [data.go.kr / 15112833](https://www.data.go.kr/data/15112833/fileData.do) — 중소벤처기업진흥공단 |
| 운영 사이트 | [joonggomall.or.kr](https://www.joonggomall.or.kr) |
| 수집일 | 2026-08-07 (파일 기준일 2025-12-31) |
| 갱신 주기 | 연 1회 — 실시간 시세가 아니라 **스냅샷** |
| 인코딩 | **cp949** (utf-8 로 열면 깨진다) |

**왜 받았나** — `12_MAINT_VALUE §9` 의 `residual_curve` 를 채울 유일한 실데이터 후보였다.
**결과는 반증이다(D74)** — 아래 §3-4 에서 직접 확인한다.

In [ ]:
CSV = EXTERNAL / "중소벤처기업진흥공단_자산거래중개장터 매물정보_20251231.csv"
print("파일:", CSV.name)
print("크기:", f"{CSV.stat().st_size:,} bytes")
print("존재:", CSV.exists())

## 2. 로드 + 기초 통계

In [ ]:
df_raw = pd.read_csv(CSV, encoding="cp949", dtype=str)
print(f"원본 shape: {df_raw.shape[0]:,}행 × {df_raw.shape[1]}열")

### 2-1. 컬럼 카탈로그 — "실제로 어떻게 생겼나"

`null%` 만 보면 안 된다. 이 데이터의 핵심 함정이 여기서 드러난다.

In [ ]:
profile(df_raw)

### 2-2. 실제 레코드 미리보기

In [ ]:
df_raw.head(5)

## 3. EDA

### 3-1. 🚨 `확인불가` 함정 — 결측률 0% 를 믿으면 안 된다

In [ ]:
print("제조년월 null 수:", df_raw["제조년월"].isna().sum(), "→ 결측률 0%처럼 보인다")
print()
print(top_values(df_raw, "제조년월", 5).to_string(index=False))
print()
n_unknown = (df_raw["제조년월"] == "확인불가").sum()
print(f"'확인불가' 문자열: {n_unknown:,}건 ({n_unknown / len(df_raw) * 100:.1f}%)")
print("→ 결측이 NaN 이 아니라 문자열로 채워져 있다. 16,011 을 분모로 쓰면 전부 틀어진다.")

### 3-2. 🚨 카테고리 문자열 — 원본 바이트를 보존해야 한다

`'환경  설비'` 는 **공백 2칸**이고 `'환경 / 설비'` 와는 **다른 값**이다.
한 칸만 틀려도 `assets` 조인이 통째로 미스난다 (`data_list.md §6-1`).
아래는 `repr()` 출력이라 공백이 그대로 보인다.

In [ ]:
print(top_values(df_raw, "카테고리1", 12).to_string(index=False))

In [ ]:
변종 = [c for c in df_raw["카테고리1"].dropna().unique() if "환경" in c]
print("'환경' 을 포함한 카테고리 값 —")
for v in 변종:
    print(f"  {v!r}  ({(df_raw['카테고리1'] == v).sum():,}건)")

### 3-3. 파싱 가능한 표본은 얼마나 되나

🚨 **`to_datetime` 을 쓰면 안 된다.** `제조년월` 에는 `2022` 처럼 **연도만 있는 값**이 섞여 있어
`pd.to_datetime` 이 조용히 실패한다. 정규식으로 앞 4자리를 뽑으면 **81건이 더 살아난다.**

In [ ]:
s = df_raw["제조년월"]
연도_dt = pd.to_datetime(s, errors="coerce").dt.year               # ❌ 순진한 방법
_y = s.str.extract(r"^(19|20)(\d{2})")
연도 = pd.to_numeric(_y[0] + _y[1], errors="coerce")                # ✅ 정규식

print(f"to_datetime 성공 : {연도_dt.notna().sum():,}건")
print(f"정규식 성공      : {연도.notna().sum():,}건")
누락 = s[연도_dt.isna() & 연도.notna()]
print(f"차이            : {len(누락):,}건 — to_datetime 이 놓친 값들")
print("  예:", [repr(v) for v in 누락.unique()[:6]], "… 연도만 있고 월이 없다")

In [ ]:
가격 = pd.to_numeric(df_raw["희망가격"], errors="coerce")

print(f"전체                    : {len(df_raw):,}건")
print(f"연도 파싱 성공          : {연도.notna().sum():,}건 ({연도.notna().mean() * 100:.1f}%)")
print(f"가격 숫자 변환 성공     : {가격.notna().sum():,}건 (100% — 0 도 숫자다)")
print(f"  그중 가격 > 0         : {(가격 > 0).sum():,}건")
print(f"  그중 가격 >= 10,000   : {(가격 >= 10_000).sum():,}건")
print()
for 라벨, m in [("가격 무관", 가격.notna()), ("가격>0", 가격 > 0), ("가격>=1만", 가격 >= 10_000)]:
    n = (연도.notna() & m).sum()
    print(f"연도 O + {라벨:9} : {n:,}건 ({n / len(df_raw) * 100:.1f}%)")
유효 = 연도.notna() & (가격 >= 10_000)
print()
print("→ '분석 가능 표본' 은 가격 기준을 어디로 잡느냐에 따라 5,475 ~ 5,911 로 달라진다.")
print("  단일 숫자로 인용하면 안 되고, 기준을 함께 적어야 한다.")
print()
print("연도 범위:", int(연도.min()), "~", int(연도.max()), "· 중앙값", int(연도.median()))

> ⚠️ **`data_list.md §6-1` 과 어긋난다.** 문서는 *"연도 파싱 성공 5,911 · 연도 O + 가격 O
> **5,396건(33.7%)**"* 이라고 적는데, 5,911 은 정규식 기준이고 5,396 은 `to_datetime` 기준(5,394)에
> 가깝다 — **두 수치가 서로 다른 파서로 산출돼 섞여 있다.** 이 노트북은 정규식으로 통일한다.

### 3-4. 이상치 — 희망가격

In [ ]:
print(가격.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).apply(lambda v: f"{v:,.0f}"))
print()
print(f"1만원 미만 더미 가격: {(가격 < 10_000).sum():,}건")
print(f"최대값             : {가격.max():,.0f}원  ← 이상치")

### 3-5. ⚠️ 잔가율 산출 가능성 — 반증 (D74)

경과연수와 가격의 상관을 직접 본다. **호가는 연차가 아니라 기계 규격이 지배한다**는 것이
`residual_curve.md §4` 의 결론이고, 아래가 그 근거다.

In [ ]:
sub = pd.DataFrame({"연도": 연도, "가격": 가격, "카테고리1": df_raw["카테고리1"]})[유효]
sub["경과연수"] = 2026 - sub["연도"]
sub = sub[(sub["경과연수"].between(0, 40)) & (sub["가격"] >= 10_000)]

print("카테고리별 경과연수 ↔ 가격 상관계수 (표본 200건 이상인 카테고리만)")
for cat, g in sub.groupby("카테고리1"):
    if len(g) >= 200:
        r = g["경과연수"].corr(g["가격"])
        print(f"  {cat!r:22} n={len(g):5,}  r={r:+.3f}")
print()
print("→ 상관이 사실상 0. 감가가 식별되지 않는다 → D74 (목업 정률 공식으로 전환)")

## 4. 전처리

원본을 파괴하지 않고 **파생 컬럼을 추가**하는 방식으로 간다.
`확인불가` 같은 원본 문자열은 지우지 않고 `_원본` 접미사로 남긴다 — 나중에 왜 결측인지
되짚을 수 있어야 한다.

In [ ]:
df = df_raw.copy()

# 1) 결측 표식 정규화 — '확인불가'·'미상'·'' 를 NA 로. 원본은 별도 컬럼에 보존
결측표식 = {"확인불가", "미상", "", "-"}
df["제조년월_원본"] = df["제조년월"]
df["제조년월"] = df["제조년월"].where(~df["제조년월"].isin(결측표식))

# 2) 타입 변환 — ⛔ to_datetime 금지. '2022' 같은 연도-only 값 81건이 날아간다 (§3-3)
_y = df["제조년월"].str.extract(r"^(19|20)(\d{2})")
df["제조년"] = pd.to_numeric(_y[0] + _y[1], errors="coerce").astype("Int64")
df["희망가격"] = pd.to_numeric(df["희망가격"], errors="coerce")
df["조회수"] = pd.to_numeric(df["조회수"], errors="coerce").astype("Int64")
df["문의갯수"] = pd.to_numeric(df["문의갯수"], errors="coerce").astype("Int64")
df["수량"] = pd.to_numeric(df["수량"], errors="coerce").astype("Int64")

# 3) 파생 — 경과연수
df["경과연수"] = (2026 - df["제조년"]).astype("Int64")

# 4) 더미 가격 플래그 (제거하지 않는다 — 판단은 소비처에서)
df["더미가격_의심"] = df["희망가격"] < 10_000

# 5) ⛔ 카테고리는 손대지 않는다 — 공백 2칸이 조인 키다 (D68 · residual_curve.md §2-6)
#    분석가능 = 연도 O + 가격 1만원 이상 (기준을 코드에 박아 둔다 — §3-3 참조)
df["분석가능"] = df["제조년"].notna() & (df["희망가격"] >= 10_000)

print(f"정제 후: {df.shape[0]:,}행 × {df.shape[1]}열 (파생 6컬럼 추가)")
print(f"분석가능 표본: {df['분석가능'].sum():,}건")

## 5. 결정 로그

각 컬럼을 **쓰는지 / 안 쓰는지 + 이유**. 이 리스트가 보고서의 원천이다.

In [ ]:
column_decisions = [
    {"column": "카테고리1", "used": True,
     "reason": "assets.category 조인 키 (D68). ⛔ 원본 바이트 유지 — '환경  설비' 공백 2칸"},
    {"column": "카테고리2", "used": True, "reason": "층화 분석용 세부 분류"},
    {"column": "제조년월", "used": True,
     "reason": "경과연수의 유일한 원천. '확인불가' 문자열을 NA 로 정규화 (63.1% 가 결측). "
               "⛔ to_datetime 금지 — '2022' 같은 연도-only 값 81건이 조용히 날아간다"},
    {"column": "희망가격", "used": True, "reason": "가격 축. 단 호가지 실거래가가 아님 — 인용 시 고지 필수"},
    {"column": "제조사", "used": True, "reason": "기술적 진부화 판별 (제조사 존속 여부)"},
    {"column": "모델명", "used": True, "reason": "동일 모델 내 연차-가격 상관 검증에 사용 (D74 근거)"},
    {"column": "제품상태", "used": True, "reason": "실사 체크리스트 그라운딩 (12 §6 물리적 상태)"},
    {"column": "가동여부", "used": True, "reason": "실사 체크리스트 그라운딩"},
    {"column": "수리가능여부", "used": True, "reason": "실사 체크리스트 그라운딩"},
    {"column": "수량", "used": True, "reason": "단가 환산 시 필요"},
    {"column": "제품명", "used": False, "reason": "자유 텍스트. 도메인 키워드 검색에만 썼고 구조화 불가"},
    {"column": "키워드", "used": False, "reason": "판매자 입력 자유 텍스트 · 결측 많음"},
    {"column": "제조국", "used": False, "reason": "'대한민국' 편중 — 변별력 없음"},
    {"column": "물건위치1", "used": False, "reason": "지역은 MaintQ 판정에 들어가지 않는다"},
    {"column": "물건위치2", "used": False, "reason": "위와 동일"},
    {"column": "거래가능지역", "used": False, "reason": "'전체' 편중 · 판정 미사용"},
    {"column": "조회수", "used": False, "reason": "수집 시점 의존 · 스냅샷이라 시계열 해석 불가"},
    {"column": "문의갯수", "used": False, "reason": "위와 동일"},
    {"column": "작성일", "used": False, "reason": "전건 2025-12-31 (파일 생성일) — 변별력 없음"},
    {"column": "판매상태", "used": False, "reason": "'판매중' 편중"},
    {"column": "상태", "used": False, "reason": "'정상' 편중 — 게시글 상태지 설비 상태가 아님"},
    {"column": "사용여부", "used": False, "reason": "'Y' 편중"},
    {"column": "가격협의가능", "used": False, "reason": "호가의 불확실성을 키울 뿐 판정에 쓸 수 없음"},
]

assert len(column_decisions) == df_raw.shape[1], \
    f"결정 로그 {len(column_decisions)}개 ≠ 원본 컬럼 {df_raw.shape[1]}개"
print(f"컬럼 결정 {len(column_decisions)}건 — 전체 컬럼 수와 일치 확인")

In [ ]:
save_decisions(
    "joonggomall_assets",
    column_decisions,
    source_url="https://www.data.go.kr/data/15112833/fileData.do",
    collected_at="2026-08-07",
    encoding="cp949",
    row_count_raw=int(df_raw.shape[0]),
    col_count_raw=int(df_raw.shape[1]),
    row_count_processed=int(df.shape[0]),
    analyzable_rows=int(df["분석가능"].sum()),
    generated_columns=["제조년월_원본", "제조년", "경과연수", "더미가격_의심", "분석가능"],
    caveats=[
        "희망가격은 호가지 실거래가가 아니다 — 인용 시 추정치 고지 필수 (D65)",
        "제조년월 결측은 NaN 이 아니라 '확인불가' 문자열이었다 (결측률 0% 함정)",
        "제조년 추출에 to_datetime 을 쓰면 연도-only 값 81건이 날아간다 — 정규식으로 뽑았다",
        "'분석 가능 표본' 은 가격 기준에 따라 5,475(>=1만) ~ 5,911(무관) 로 달라진다. "
        "이 노트북은 >=1만 기준을 쓴다",
        "잔가율 값 원천으로는 반증됐다 (D74) — 연차-가격 상관 ≈ 0",
        "카테고리1 은 원본 바이트 유지 — '환경  설비'(공백 2칸) ≠ '환경 / 설비'",
    ],
    discrepancy_note=(
        "[해결됨 2026-08-12] data_list.md §6-1 이 '연도 5,911 · 연도+가격 5,396' 으로 "
        "서로 다른 파서의 값을 섞어 쓰고 있었다(5,911=정규식 / 5,396=to_datetime 계열, 재현값 5,394). "
        "D72 가 이미 '연도 파싱 ∧ 가격≥10,000 → 5,475' 로 필터를 고정해 둔 상태였고, "
        "이 노트북의 독립 재측정이 D72 의 세 값(5,911/5,685/5,475)과 정확히 일치했다. "
        "→ data_list.md 를 D72 기준으로 정정 완료. "
        "아직 옛 5,396 이 남은 곳: docs/status/maintq-data-map.html · maintq-diagrams.html · "
        "docs/sessions/2026-08-07.md (D68 본문은 결정 기록이라 보존)."
    ),
)

## 6. 정제 데이터 저장

In [ ]:
save_processed("joonggomall_assets", df)